In [1]:
# ============================================================
#  SIMPENAN - OOP: ENKAPSULASI (Encapsulation)
#  Sistem Informasi Penyakit Tanaman Nasional
#  Konteks: Data sensitif seperti password, PIN, dan hasil
#           diagnosis dilindungi dari akses langsung luar class
# ============================================================

import hashlib   # Library untuk hashing password (pengganti password_hash() PHP)
import datetime  # Library untuk mencatat waktu login dan aktivitas

# ─────────────────────────────────────────────────────────────
# KELAS 1: AkunPengguna
# Melindungi data sensitif: password & riwayat login
# Setara dengan tabel 'users' di database SIMPENAN
# ─────────────────────────────────────────────────────────────

class AkunPengguna:
    """
    Class AkunPengguna - Enkapsulasi data akun SIMPENAN.

    ATRIBUT PRIVATE (tidak bisa diakses langsung dari luar class):
      __password      → password di-hash, tidak boleh terekspos
      __role          → role user/admin, tidak boleh diubah sembarangan
      __riwayat_login → log aktivitas, hanya bisa ditambah, tidak bisa dihapus

    Setara dengan logika di prosesLogin.php dan session cookie SIMPENAN.
    """

    def __init__(self, username, email, password_asli, role="user"):
        # PUBLIC ATTRIBUTE: username dan email boleh diakses langsung
        self.username = username   # Nama pengguna untuk login
        self.email    = email      # Email terdaftar

        # PRIVATE ATTRIBUTE: diberi prefix __ (double underscore)
        # Atribut ini TIDAK BISA diakses dari luar class: obj.__password → ERROR!
        self.__password      = self.__hash_password(password_asli)  # Simpan versi hash-nya
        self.__role          = role                                  # 'user' atau 'admin'
        self.__riwayat_login = []   # List kosong untuk mencatat waktu login

    # ── PRIVATE METHOD: Hashing password ──────────────────────
    def __hash_password(self, password):
        # Method ini PRIVATE (awalan __), hanya bisa dipanggil dari dalam class
        # Setara dengan password_hash() di PHP/SIMPENAN
        # Menggunakan SHA-256 untuk keamanan (dalam produksi gunakan bcrypt)
        return hashlib.sha256(password.encode()).hexdigest()  # Kembalikan string hash

    # ── GETTER: Mendapatkan nilai atribut private ──────────────
    def get_role(self):
        # Getter untuk __role: hanya memberikan BACA, tidak bisa UBAH
        return self.__role   # Kembalikan nilai role saat ini

    def get_jumlah_login(self):
        # Getter untuk jumlah login: mengembalikan panjang list riwayat
        return len(self.__riwayat_login)   # Jumlah berapa kali user sudah login

    def get_riwayat_login(self):
        # Getter untuk melihat riwayat login (hanya baca, tidak bisa ubah)
        # Mengembalikan COPY list agar list asli tidak bisa dimanipulasi
        return list(self.__riwayat_login)   # Buat salinan list, kembalikan salinannya

    # ── SETTER: Mengubah nilai atribut private dengan validasi ─
    def set_role(self, role_baru, konfirmasi_admin=False):
        """
        Setter untuk __role dengan validasi ketat.
        Mengubah role hanya boleh dilakukan jika ada konfirmasi admin.
        Ini mencegah user biasa mengubah role mereka sendiri jadi admin.
        """
        role_valid = ["user", "admin"]   # Daftar role yang sah di SIMPENAN

        if role_baru not in role_valid:
            # Validasi: role harus salah satu dari yang terdefinisi
            print(f"❌ Error: Role '{role_baru}' tidak valid! Pilihan: {role_valid}")
            return False  # Gagal mengubah role

        if not konfirmasi_admin:
            # Validasi: perubahan role butuh konfirmasi dari admin
            print("❌ Error: Mengubah role membutuhkan konfirmasi admin!")
            return False  # Gagal tanpa konfirmasi

        # Jika semua validasi lolos, barulah ubah role
        self.__role = role_baru   # Ubah nilai private __role
        print(f"✅ Role '{self.username}' berhasil diubah menjadi '{role_baru}'")
        return True

    def ganti_password(self, password_lama, password_baru):
        """
        Method untuk mengganti password dengan verifikasi password lama.
        Validasi: password baru minimal 8 karakter.
        Ini setara dengan fitur reset password di SIMPENAN.
        """
        # Verifikasi password lama dulu sebelum mengizinkan penggantian
        if self.__hash_password(password_lama) != self.__password:
            print("❌ Gagal: Password lama tidak cocok!")
            return False   # Tolak perubahan jika password lama salah

        if len(password_baru) < 8:
            # Validasi: password baru harus cukup kuat (minimal 8 karakter)
            print("❌ Gagal: Password baru harus minimal 8 karakter!")
            return False

        # Jika semua validasi lolos, simpan hash password baru
        self.__password = self.__hash_password(password_baru)
        print(f"✅ Password '{self.username}' berhasil diperbarui.")
        return True

    # ── METHOD: Login dengan verifikasi ───────────────────────
    def login(self, username_input, password_input):
        """
        Method untuk proses login ke SIMPENAN.
        Setara dengan logika di prosesLogin.php.
        Password tidak pernah dibandingkan langsung — selalu via hash.
        """
        # Bandingkan username dan hash password input dengan yang tersimpan
        if (self.username == username_input and
                self.__hash_password(password_input) == self.__password):
            # Login berhasil: catat waktu login ke riwayat (PRIVATE)
            waktu_login = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            self.__riwayat_login.append(waktu_login)   # Tambah ke list private
            print(f"✅ Login berhasil! Selamat datang di SIMPENAN, {self.username}!")
            print(f"   Role    : {self.__role}")         # Tampilkan role
            print(f"   Waktu   : {waktu_login}")         # Tampilkan waktu login
            return True

        else:
            # Login gagal: setara dengan alert('Username atau Password Salah!')
            print("❌ Login gagal: Username atau password salah!")
            return False

    def __str__(self):
        # Representasi string — PASSWORD TIDAK DITAMPILKAN sama sekali!
        return f"AkunPengguna(username='{self.username}', role='{self.__role}')"

In [2]:
# ─────────────────────────────────────────────────────────────
# KELAS 2: DataDiagnosis
# Melindungi data sensitif: hasil diagnosis penyakit tanaman
# Data ini bersifat rahasia (hanya dokter/penyuluh yang boleh ubah)
# ─────────────────────────────────────────────────────────────

class DataDiagnosis:
    """
    Class DataDiagnosis - Enkapsulasi hasil diagnosis penyakit tanaman.

    ATRIBUT PRIVATE (rahasia, tidak boleh diubah sembarangan):
      __nama_tanaman    → data yang sudah terverifikasi
      __hasil_diagnosis → diagnosis resmi dari sistem
      __skor_keparahan  → nilai 0-100, hanya boleh diisi dalam rentang valid
      __catatan_khusus  → catatan tambahan dari penyuluh pertanian

    Analoginya: seperti rekam medis pasien — hanya dokter yang boleh
    mengisi/mengubah, pasien hanya boleh melihat.
    """

    def __init__(self, id_laporan, nama_petani, nama_tanaman):
        # PUBLIC: informasi dasar yang boleh diakses umum
        self.id_laporan  = id_laporan    # ID laporan di sistem SIMPENAN
        self.nama_petani = nama_petani   # Nama petani pelapor

        # PROTECTED ATTRIBUTE: prefix _ (satu underscore)
        # Disarankan internal/turunan, tapi masih bisa diakses jika perlu
        self._status = "MENUNGGU"   # Status: MENUNGGU / DIPROSES / SELESAI

        # PRIVATE ATTRIBUTE: prefix __ (dua underscore)
        # Tidak bisa diakses langsung dari luar class
        self.__nama_tanaman    = nama_tanaman  # Nama tanaman yang didiagnosis
        self.__hasil_diagnosis = None          # Belum ada diagnosis (None = belum diisi)
        self.__skor_keparahan  = None          # Belum ada skor (0-100)
        self.__catatan_khusus  = ""            # Catatan tambahan penyuluh

    # ── GETTER: Mengambil nilai atribut private ────────────────
    def get_nama_tanaman(self):
        # Getter sederhana untuk nama tanaman
        return self.__nama_tanaman   # Kembalikan nilai private

    def get_hasil_diagnosis(self):
        # Getter dengan pengecekan: apakah diagnosis sudah diisi?
        if self.__hasil_diagnosis is None:
            return "Belum ada diagnosis"   # Kembalikan pesan default jika belum diisi
        return self.__hasil_diagnosis       # Kembalikan hasil diagnosis

    def get_skor_keparahan(self):
        # Getter untuk skor keparahan
        if self.__skor_keparahan is None:
            return "Belum dinilai"   # Skor belum diisi oleh penyuluh
        return self.__skor_keparahan  # Kembalikan nilai skor

    def get_level_keparahan(self):
        """
        Getter dengan logika tambahan: konversi skor numerik ke level teks.
        Skor 0-100 → Ringan / Sedang / Parah / Kritis
        """
        skor = self.__skor_keparahan   # Ambil nilai private

        if skor is None:
            return "Belum dinilai"   # Belum ada skor sama sekali
        elif skor >= 75:
            return "🔴 KRITIS"      # Keparahan sangat tinggi
        elif skor >= 50:
            return "🟠 PARAH"       # Keparahan tinggi
        elif skor >= 25:
            return "🟡 SEDANG"      # Keparahan menengah
        else:
            return "🟢 RINGAN"      # Keparahan rendah

    # ── SETTER: Mengubah nilai dengan validasi ─────────────────
    def set_hasil_diagnosis(self, nama_penyakit, skor_keparahan, catatan=""):
        """
        Setter untuk mengisi hasil diagnosis.
        Hanya bisa dipanggil oleh penyuluh/admin melalui method ini.
        Validasi: skor keparahan harus 0-100, nama penyakit tidak boleh kosong.
        """
        if not nama_penyakit or not nama_penyakit.strip():
            # Validasi: nama penyakit tidak boleh string kosong
            print("❌ Error: Nama penyakit tidak boleh kosong!")
            return False

        if not isinstance(skor_keparahan, (int, float)):
            # Validasi: skor harus berupa angka (integer atau float)
            print("❌ Error: Skor keparahan harus berupa angka!")
            return False

        if not (0 <= skor_keparahan <= 100):
            # Validasi: skor harus dalam rentang 0-100
            print(f"❌ Error: Skor keparahan '{skor_keparahan}' harus antara 0-100!")
            return False

        # Semua validasi lolos: isi atribut private
        self.__hasil_diagnosis = nama_penyakit.strip()   # Simpan nama penyakit
        self.__skor_keparahan  = skor_keparahan           # Simpan skor
        self.__catatan_khusus  = catatan                  # Simpan catatan penyuluh
        self._status           = "SELESAI"                # Update status (protected)
        print(f"✅ Diagnosis berhasil diinput untuk laporan #{self.id_laporan}")
        return True

    def update_catatan(self, catatan_baru):
        """
        Setter khusus untuk memperbarui catatan saja (tanpa ubah diagnosis).
        Catatan bisa diperbarui tanpa harus mengulang seluruh diagnosis.
        """
        if not catatan_baru.strip():
            # Validasi: catatan tidak boleh kosong jika ingin diperbarui
            print("❌ Error: Catatan tidak boleh kosong!")
            return False

        self.__catatan_khusus = catatan_baru.strip()   # Perbarui catatan
        print(f"✅ Catatan laporan #{self.id_laporan} berhasil diperbarui.")
        return True

    # ── METHOD: Tampilkan ringkasan diagnosis ─────────────────
    def tampilkan_ringkasan(self):
        """
        Method untuk menampilkan ringkasan diagnosis.
        Data sensitif ditampilkan terkontrol melalui getter,
        bukan diakses langsung dari luar.
        """
        print(f"\n{'=' * 50}")
        print(f"  📄 LAPORAN DIAGNOSIS #{self.id_laporan}")
        print(f"{'=' * 50}")
        print(f"  Petani       : {self.nama_petani}")         # Public attribute
        print(f"  Tanaman      : {self.get_nama_tanaman()}")  # Via getter
        print(f"  Status       : {self._status}")             # Protected attribute
        print(f"  Diagnosis    : {self.get_hasil_diagnosis()}")         # Via getter
        print(f"  Skor         : {self.get_skor_keparahan()}/100")      # Via getter
        print(f"  Level        : {self.get_level_keparahan()}")         # Via getter logic
        if self.__catatan_khusus:
            # Tampilkan catatan hanya jika ada isinya
            print(f"  Catatan      : {self.__catatan_khusus}")

    def __str__(self):
        # Representasi string — skor keparahan tidak ditampilkan langsung
        return f"DataDiagnosis(#{self.id_laporan}, '{self.nama_petani}')"

In [3]:
# ════════════════════════════════════════════════════════════
#  DEMO: Enkapsulasi SIMPENAN
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print("=" * 60)
    print("   SIMPENAN - Demo Enkapsulasi")
    print("   Melindungi data sensitif pengguna & diagnosis")
    print("=" * 60)

    # ──────────────────────────────────────────────
    # DEMO 1: AkunPengguna — Proteksi Password & Role
    # ──────────────────────────────────────────────
    print("\n🔐 DEMO 1: Akun Pengguna SIMPENAN")
    print("-" * 50)

    # Membuat objek akun pengguna
    akun_petani = AkunPengguna(
        username     = "siti_petani",
        email        = "siti@gmail.com",
        password_asli= "Sawah2024!",
        role         = "user"
    )

    # Coba akses atribut private langsung → akan gagal!
    print("\n[Uji Coba Akses Langsung ke Atribut Private]")
    try:
        print(akun_petani.__password)   # Ini akan error: AttributeError
    except AttributeError as e:
        # Enkapsulasi berhasil: atribut private tidak bisa diakses dari luar
        print(f"❌ Akses ditolak! Error: {type(e).__name__}")
        print("   → Atribut __password terlindungi oleh enkapsulasi!")

    # Akses role yang benar: melalui GETTER
    print(f"\n[Getter] Role pengguna : {akun_petani.get_role()}")   # Via getter, bukan langsung

    # Proses login: password asli dibandingkan secara aman di dalam method
    print("\n[Test Login]")
    akun_petani.login("siti_petani", "passwordSalah")   # Percobaan gagal
    akun_petani.login("siti_petani", "Sawah2024!")      # Login berhasil

    # Cek riwayat login via getter
    print(f"\n[Getter] Jumlah login : {akun_petani.get_jumlah_login()} kali")
    print(f"[Getter] Riwayat     : {akun_petani.get_riwayat_login()}")

    # Ganti password dengan validasi
    print("\n[Ganti Password]")
    akun_petani.ganti_password("passwordSalah", "BuatBaru!")   # Gagal: password lama salah
    akun_petani.ganti_password("Sawah2024!", "Padi")            # Gagal: terlalu pendek
    akun_petani.ganti_password("Sawah2024!", "NilaiSkor99!")    # Berhasil!

    # Ubah role dengan setter — perlu konfirmasi admin
    print("\n[Setter Role]")
    akun_petani.set_role("admin")                       # Gagal: tidak ada konfirmasi
    akun_petani.set_role("superuser")                   # Gagal: role tidak valid
    akun_petani.set_role("admin", konfirmasi_admin=True) # Berhasil!
    print(f"[Getter] Role sekarang: {akun_petani.get_role()}")

    print(f"\n{akun_petani}")   # Panggil __str__

   SIMPENAN - Demo Enkapsulasi
   Melindungi data sensitif pengguna & diagnosis

🔐 DEMO 1: Akun Pengguna SIMPENAN
--------------------------------------------------

[Uji Coba Akses Langsung ke Atribut Private]
❌ Akses ditolak! Error: AttributeError
   → Atribut __password terlindungi oleh enkapsulasi!

[Getter] Role pengguna : user

[Test Login]
❌ Login gagal: Username atau password salah!
✅ Login berhasil! Selamat datang di SIMPENAN, siti_petani!
   Role    : user
   Waktu   : 2026-05-01 07:04:47

[Getter] Jumlah login : 1 kali
[Getter] Riwayat     : ['2026-05-01 07:04:47']

[Ganti Password]
❌ Gagal: Password lama tidak cocok!
❌ Gagal: Password baru harus minimal 8 karakter!
✅ Password 'siti_petani' berhasil diperbarui.

[Setter Role]
❌ Error: Mengubah role membutuhkan konfirmasi admin!
❌ Error: Role 'superuser' tidak valid! Pilihan: ['user', 'admin']
✅ Role 'siti_petani' berhasil diubah menjadi 'admin'
[Getter] Role sekarang: admin

AkunPengguna(username='siti_petani', role='admin')

In [4]:
    # ──────────────────────────────────────────────
    # DEMO 2: DataDiagnosis — Proteksi Data Rekam Medis Tanaman
    # ──────────────────────────────────────────────
    print("\n\n🌾 DEMO 2: Data Diagnosis Penyakit Tanaman")
    print("-" * 50)

    # Buat laporan diagnosis baru
    laporan_1 = DataDiagnosis(
        id_laporan   = "SMP-2026-001",
        nama_petani  = "Siti Rahayu",
        nama_tanaman = "Padi Ciherang"
    )

    # Tampilkan sebelum diagnosis diisi
    print("\n[Sebelum Diagnosis:]")
    laporan_1.tampilkan_ringkasan()

    # Coba akses langsung atribut private → gagal!
    print("\n[Uji Coba Akses Langsung]")
    try:
        print(laporan_1.__skor_keparahan)   # Ini akan error
    except AttributeError as e:
        print(f"❌ Akses ditolak! __skor_keparahan tidak bisa diakses langsung!")

    # Setter: isi hasil diagnosis dengan validasi
    print("\n[Setter: Input Hasil Diagnosis]")
    laporan_1.set_hasil_diagnosis("", 80)          # Gagal: nama kosong
    laporan_1.set_hasil_diagnosis("Blast Padi", 150)  # Gagal: skor > 100
    laporan_1.set_hasil_diagnosis(               # Berhasil: semua valid
        "Blast Padi (Pyricularia oryzae)",
        skor_keparahan = 72,
        catatan        = "Perlu penanganan segera, terutama pada petak A-3 dan B-1"
    )

    # Perbarui catatan saja
    laporan_1.update_catatan("Sudah disemprot fungisida, monitoring seminggu lagi")

    # Tampilkan setelah diagnosis diisi
    print("\n[Setelah Diagnosis:]")
    laporan_1.tampilkan_ringkasan()



🌾 DEMO 2: Data Diagnosis Penyakit Tanaman
--------------------------------------------------

[Sebelum Diagnosis:]

  📄 LAPORAN DIAGNOSIS #SMP-2026-001
  Petani       : Siti Rahayu
  Tanaman      : Padi Ciherang
  Status       : MENUNGGU
  Diagnosis    : Belum ada diagnosis
  Skor         : Belum dinilai/100
  Level        : Belum dinilai

[Uji Coba Akses Langsung]
❌ Akses ditolak! __skor_keparahan tidak bisa diakses langsung!

[Setter: Input Hasil Diagnosis]
❌ Error: Nama penyakit tidak boleh kosong!
❌ Error: Skor keparahan '150' harus antara 0-100!
✅ Diagnosis berhasil diinput untuk laporan #SMP-2026-001
✅ Catatan laporan #SMP-2026-001 berhasil diperbarui.

[Setelah Diagnosis:]

  📄 LAPORAN DIAGNOSIS #SMP-2026-001
  Petani       : Siti Rahayu
  Tanaman      : Padi Ciherang
  Status       : SELESAI
  Diagnosis    : Blast Padi (Pyricularia oryzae)
  Skor         : 72/100
  Level        : 🟠 PARAH
  Catatan      : Sudah disemprot fungisida, monitoring seminggu lagi


In [5]:
    # ──────────────────────────────────────────────
    # RANGKUMAN: Jenis Atribut yang Digunakan
    # ──────────────────────────────────────────────
    print("\n\n" + "=" * 60)
    print("  📚 RINGKASAN ENKAPSULASI SIMPENAN:")
    print("=" * 60)
    print("  Tipe       | Penulisan | Contoh di SIMPENAN")
    print("  -----------|-----------|------------------------")
    print("  Public     | nama      | username, email, id_laporan")
    print("  Protected  | _nama     | _status (boleh akses internal)")
    print("  Private    | __nama    | __password, __skor_keparahan")
    print()
    print("  Akses data private hanya lewat: Getter & Setter!")
    print("  Perubahan data private divalidasi di dalam class!")



  📚 RINGKASAN ENKAPSULASI SIMPENAN:
  Tipe       | Penulisan | Contoh di SIMPENAN
  -----------|-----------|------------------------
  Public     | nama      | username, email, id_laporan
  Protected  | _nama     | _status (boleh akses internal)
  Private    | __nama    | __password, __skor_keparahan

  Akses data private hanya lewat: Getter & Setter!
  Perubahan data private divalidasi di dalam class!
